# Graph Sparsification Experiments

Systematic comparison of **Metric Backbone** and **Effective Resistance** sparsification
across configuration models with various degree and weight distributions.
Beta is auto-calibrated for each graph so that the SIR epidemic is in an interesting regime.

In [ ]:
import sys, os
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
src_path = os.path.join(repo_root, 'src', 'python')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

In [ ]:
import numpy as np
from scipy import sparse
import matplotlib.pyplot as plt
import time

from graph_sparsification.generators import configuration_model
from graph_sparsification.sparsifiers import metric_backbone, effective_resistance_sparsify
from graph_sparsification.sir import sir_monte_carlo, calibrate_beta
from graph_sparsification.visualization import (
    plot_adjacency_comparison,
    plot_multi_infection_comparison,
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

try:
    from graph_sparsification._sir_cpp import sir_simulation_cpp
    print('C++ SIR backend: available')
except ImportError:
    print('C++ SIR backend: NOT available (using Python fallback)')

## 1. Distribution definitions

In [ ]:
# ── Degree distributions ──────────────────────────────────────────────

degree_distributions = {
    'Unif(1,50)':  lambda n, rng: rng.integers(1, 51, size=n),
    'Unif(1,100)': lambda n, rng: rng.integers(1, 101, size=n),
    'Exp(30)':     lambda n, rng: np.ceil(rng.exponential(30, size=n)).astype(int),
    'Exp(60)':     lambda n, rng: np.ceil(rng.exponential(60, size=n)).astype(int),
    'LogN(3.26,0.66)': lambda n, rng: np.ceil(rng.lognormal(3.26, 0.66, size=n)).astype(int),
    'LogN(3.26,2)':    lambda n, rng: np.ceil(rng.lognormal(3.26, 2.0, size=n)).astype(int),
    'Pareto(2.5,20)':  lambda n, rng: np.ceil((rng.pareto(2.5, size=n) + 1) * 20).astype(int),
    'Pareto(1.5,30)':  lambda n, rng: np.ceil((rng.pareto(1.5, size=n) + 1) * 30).astype(int),
}

# ── Weight (distance/cost) distributions ──────────────────────────────

weight_distributions = {
    'Exp(1/30)':       lambda m, rng: rng.exponential(30.0, size=m),   # scale = 1/rate
    'Exp(1)':          lambda m, rng: rng.exponential(1.0, size=m),
    'Exp(30)':         lambda m, rng: rng.exponential(1/30, size=m),
    'LogN(2,1)':       lambda m, rng: rng.lognormal(2.0, 1.0, size=m),
    'LogLogN(1.2,0.4)': lambda m, rng: np.exp(rng.lognormal(1.2, 0.4, size=m)),
    'LogLogN(1.2,0.8)': lambda m, rng: np.exp(rng.lognormal(1.2, 0.8, size=m)),
    'LogLogN(2,0.8)':   lambda m, rng: np.exp(rng.lognormal(2.0, 0.8, size=m)),
}

print(f'{len(degree_distributions)} degree distributions × {len(weight_distributions)} weight distributions')
print(f'= {len(degree_distributions) * len(weight_distributions)} configurations')

## 2. Experiment parameters

In [ ]:
N_NODES = 200          # nodes per graph
N_SIR_RUNS = 200       # Monte Carlo SIR runs per graph
N_CAL_RUNS = 30        # calibration runs (fewer, for speed)
GAMMA = 1.0            # recovery rate
SEED = 42

## 3. Main loop

In [ ]:
all_results = []
master_rng = np.random.default_rng(SEED)

for deg_name, deg_sampler in degree_distributions.items():
    for wt_name, wt_sampler in weight_distributions.items():
        label = f'{deg_name}  |  {wt_name}'
        run_seed = int(master_rng.integers(0, 2**31))
        rng = np.random.default_rng(run_seed)

        # ── Generate graph ────────────────────────────────────────────
        W = configuration_model(N_NODES, deg_sampler, wt_sampler, rng=rng)
        n_edges = sparse.triu(W).nnz

        if n_edges < 10:
            print(f'SKIP {label}: only {n_edges} edges')
            continue

        degrees = np.diff(W.indptr)  # actual degree per node
        weights = W.data

        print(f'\n{"="*72}')
        print(f'{label}')
        print(f'  edges={n_edges}, '
              f'deg: mean={degrees.mean():.1f} med={np.median(degrees):.0f} max={degrees.max()}, '
              f'wt: mean={weights.mean():.2f} med={np.median(weights):.2f}')

        # ── Calibrate beta ────────────────────────────────────────────
        beta, cal_info = calibrate_beta(
            W, gamma=GAMMA, target_mean_infection=0.5, target_range=(0.3, 0.7),
            n_calibration_runs=N_CAL_RUNS, rng=rng, verbose=True,
        )

        # ── Sparsify ──────────────────────────────────────────────────
        t0 = time.time()
        W_mbb = metric_backbone(W)
        t_mbb = time.time() - t0
        n_mbb = sparse.triu(W_mbb).nnz

        effr_frac = max(n_mbb / n_edges, 0.05)
        t0 = time.time()
        W_effr = effective_resistance_sparsify(W, fraction=effr_frac, rng=rng)
        t_effr = time.time() - t0
        n_effr = sparse.triu(W_effr).nnz

        print(f'  MBB:  {n_mbb} edges ({n_mbb/n_edges*100:.1f}%) [{t_mbb:.1f}s]')
        print(f'  EffR: {n_effr} edges ({n_effr/n_edges*100:.1f}%) [{t_effr:.1f}s]')

        # ── SIR Monte Carlo ───────────────────────────────────────────
        initial = [int(np.argmax(np.array(W.sum(axis=1)).ravel()))]

        sir_orig = sir_monte_carlo(W,      beta, GAMMA, initial, n_runs=N_SIR_RUNS, rng=rng)
        sir_mbb  = sir_monte_carlo(W_mbb,  beta, GAMMA, initial, n_runs=N_SIR_RUNS, rng=rng)
        sir_effr = sir_monte_carlo(W_effr, beta, GAMMA, initial, n_runs=N_SIR_RUNS, rng=rng)

        # ── Plots ─────────────────────────────────────────────────────
        fig = plot_adjacency_comparison(W, W_mbb,
                                        labels=('Original', 'Metric Backbone'))
        fig.suptitle(label, fontsize=13, y=1.02)
        plt.show()

        fig = plot_multi_infection_comparison(
            sir_orig['infection_prob'],
            [sir_mbb['infection_prob'], sir_effr['infection_prob']],
            ['Metric Backbone', 'Eff. Resistance'],
        )
        fig.suptitle(f'{label}  (beta={beta:.4f})', fontsize=13, y=1.02)
        plt.show()

        # ── Store result ──────────────────────────────────────────────
        def _r2(po, ps):
            m = np.isfinite(po) & np.isfinite(ps)
            po, ps = po[m], ps[m]
            ss_tot = np.sum((po - po.mean())**2)
            return 1 - np.sum((po - ps)**2) / ss_tot if ss_tot > 0 else 0.0

        all_results.append({
            'deg': deg_name, 'wt': wt_name, 'label': label,
            'n_edges': n_edges, 'beta': beta,
            'mbb_pct': n_mbb / n_edges * 100,
            'effr_pct': n_effr / n_edges * 100,
            'r2_mbb':  _r2(sir_orig['infection_prob'], sir_mbb['infection_prob']),
            'r2_effr': _r2(sir_orig['infection_prob'], sir_effr['infection_prob']),
            'mean_inf_orig': sir_orig['infection_prob'].mean(),
        })

## 4. Summary table

In [ ]:
print(f"{'Degree':<18} {'Weights':<18} {'edges':>6} {'beta':>8} "
      f"{'MBB%':>6} {'EffR%':>6} {'R² MBB':>7} {'R² EffR':>8}")
print('-' * 95)
for r in all_results:
    print(f"{r['deg']:<18} {r['wt']:<18} {r['n_edges']:>6} {r['beta']:>8.4f} "
          f"{r['mbb_pct']:>5.1f}% {r['effr_pct']:>5.1f}% "
          f"{r['r2_mbb']:>7.3f} {r['r2_effr']:>8.3f}")

## 5. Aggregate comparison plot

In [ ]:
labels = [r['label'] for r in all_results]
r2_mbb  = [r['r2_mbb']  for r in all_results]
r2_effr = [r['r2_effr'] for r in all_results]

y = np.arange(len(labels))
h = 0.35

fig, ax = plt.subplots(figsize=(10, max(5, len(labels) * 0.45)))
ax.barh(y + h/2, r2_mbb,  h, label='Metric Backbone',     color='steelblue')
ax.barh(y - h/2, r2_effr, h, label='Effective Resistance', color='coral')
ax.set_yticks(y)
ax.set_yticklabels(labels, fontsize=8)
ax.set_xlabel('$R^2$ (infection probability)', fontsize=12)
ax.set_title('Sparsifier fidelity across degree × weight distributions', fontsize=13)
ax.legend(fontsize=10)
ax.set_xlim(0, 1.05)
ax.axvline(1.0, color='gray', ls='--', alpha=0.3)
ax.invert_yaxis()
plt.tight_layout()
plt.show()